## 1. Configuration

In [ ]:
VS_ENDPOINT_NAME = "nyaya_vs_endpoint"
VS_INDEX_NAME = "main.india_legal.legal_rag_corpus_index"
SOURCE_TABLE = "main.india_legal.legal_rag_corpus"
# Databricks-managed embedding model (no need to run sentence-transformers in the app)
EMBEDDING_MODEL = "databricks-bge-large-en"
PRIMARY_KEY = "chunk_id"
EMBEDDING_SOURCE_COLUMN = "text"


## 2. Create Vector Search Endpoint

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.vectorsearch import EndpointType
import time

w = WorkspaceClient()

# Check if endpoint already exists
existing = [ep for ep in w.vector_search_endpoints.list_endpoints() if ep.name == VS_ENDPOINT_NAME]
if existing:
    ep_status = getattr(existing[0], "status", None) or getattr(existing[0], "endpoint_status", None)
    print(f"Endpoint '{VS_ENDPOINT_NAME}' already exists (status: {ep_status})")
else:
    print(f"Creating endpoint '{VS_ENDPOINT_NAME}'...")
    w.vector_search_endpoints.create_endpoint(
        name=VS_ENDPOINT_NAME,
        endpoint_type=EndpointType.STANDARD,
    )
    print("Endpoint creation started.")


## 3. Wait for Endpoint to be ONLINE

In [ ]:
print(f"Waiting for endpoint '{VS_ENDPOINT_NAME}' to be ONLINE...")
for i in range(60):  # up to 10 minutes
    ep = w.vector_search_endpoints.get_endpoint(VS_ENDPOINT_NAME)
    # SDK versions vary: try .status, .endpoint_status, or inspect the object
    status = getattr(ep, "status", None) or getattr(ep, "endpoint_status", None) or str(ep)
    print(f"  [{i * 10}s] Status: {status}")
    if "ONLINE" in str(status).upper():
        print("Endpoint is ONLINE!")
        break
    time.sleep(10)
else:
    print("WARNING: Endpoint not ONLINE after 10 minutes. Check the UI.")

## 4. Verify Source Table

In [ ]:
count = spark.table(SOURCE_TABLE).count()
print(f"Source table '{SOURCE_TABLE}' has {count} rows")
display(spark.table(SOURCE_TABLE).limit(3))

## 4b. Enable Change Data Feed

Delta Sync indexes require Change Data Feed (CDF) on the source table.

In [ ]:

spark.sql(f"ALTER TABLE {SOURCE_TABLE} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
print(f"Change Data Feed enabled on {SOURCE_TABLE}")

## 5. Create Delta Sync Index with Managed Embeddings

The index auto-computes embeddings from the `text` column using
`databricks-bge-large-en`. No need to pre-compute or store embeddings.

In [ ]:

from databricks.sdk.service.vectorsearch import (
    DeltaSyncVectorIndexSpecRequest,
    EmbeddingSourceColumn,
    VectorIndexType,
    PipelineType,
)

# Check if index already exists
try:
    existing_idx = w.vector_search_indexes.get_index(VS_INDEX_NAME)
    idx_status = getattr(existing_idx, "status", None) or getattr(existing_idx, "index_status", None)
    print(f"Index '{VS_INDEX_NAME}' already exists (status: {idx_status})")
except Exception:
    print(f"Creating Delta Sync index '{VS_INDEX_NAME}'...")
    w.vector_search_indexes.create_index(
        name=VS_INDEX_NAME,
        endpoint_name=VS_ENDPOINT_NAME,
        primary_key=PRIMARY_KEY,
        index_type=VectorIndexType.DELTA_SYNC,
        delta_sync_index_spec=DeltaSyncVectorIndexSpecRequest(
            source_table=SOURCE_TABLE,
            embedding_source_columns=[
                EmbeddingSourceColumn(
                    name=EMBEDDING_SOURCE_COLUMN,
                    embedding_model_endpoint_name=EMBEDDING_MODEL,
                )
            ],
            pipeline_type=PipelineType.TRIGGERED,
        ),
    )
    print("Index creation started.")

## 6. Trigger Sync and Wait

In [ ]:
print("Triggering index sync...")
try:
    w.vector_search_indexes.sync_index(VS_INDEX_NAME)
except Exception as e:
    print(f"Sync trigger note: {e}")

print(f"Waiting for index '{VS_INDEX_NAME}' to be ready...")
for i in range(60):
    try:
        idx = w.vector_search_indexes.get_index(VS_INDEX_NAME)
        status = idx.status
        print(f"  [{i * 10}s] Status: {status}")
        if "ONLINE" in str(status) or "READY" in str(status):
            print("Index is ready!")
            break
    except Exception as e:
        print(f"  [{i * 10}s] Waiting... ({e})")
    time.sleep(10)

## 7. Smoke Test — Similarity Search

In [ ]:
test_query = "What is theft under BNS?"
print(f"Testing query: '{test_query}'")

results = w.vector_search_indexes.query_index(
    index_name=VS_INDEX_NAME,
    columns=["chunk_id", "text", "title", "source", "doc_type"],
    query_text=test_query,
    num_results=5,
)

# Response is a dataclass — use as_dict() to get a plain dict, or access attributes directly.
resp = results.as_dict() if hasattr(results, "as_dict") else results
if isinstance(resp, dict):
    data_array = resp.get("result", {}).get("data_array", [])
else:
    data_array = getattr(getattr(resp, "result", None), "data_array", []) or []

print(f"\nResults ({len(data_array)} rows):")
for row in data_array:
    print(f"  {row[0]}: {row[2]} ({row[3]})")